# 82 — Documentary Noncompliance Warehouse

Builds a permit-year noncompliance evidence table from CEMS hits and documentary flags.

In [ ]:
import os, io, re, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

phase80_cfg = load_yaml(CONFIGS / "phase80.yml")
permit_cfg = load_yaml(CONFIGS / "permit_registry.yml")
run_cfg = load_yaml(CONFIGS / "run.yml")

In [ ]:
step = "82_documentary_noncompliance_warehouse"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

vocab, meta_vocab = safe_read_csv(DATA / "cems_vocab_hits.csv")
blocks, meta_blocks = safe_read_csv(DATA / "cems_table_blocks_hits.csv")
registry, _ = safe_read_csv(OUTPUTS / "80_master_permit_registry" / "master_permit_registry.csv")

frames = []
if not vocab.empty:
    v = vocab.copy()
    v["source_type"] = "vocab"
    frames.append(v)
if not blocks.empty:
    b = blocks.copy()
    b["source_type"] = "block"
    frames.append(b)
if not frames:
    raise FileNotFoundError("Need CEMS vocab or block hits")

combined = pd.concat(frames, ignore_index=True, sort=False)
combined["permit_id"] = combined.get("permit_id", "").astype(str).str.strip()
combined["report_year"] = pd.to_numeric(combined.get("report_year", combined.get("year", np.nan)), errors="coerce")
combined["text_blob"] = (
    combined.get("key", "").astype(str) + " " +
    combined.get("value_text", combined.get("text", combined.get("block_text",""))).astype(str)
).str.lower()
combined["noncompliance_flag"] = combined["text_blob"].str.contains("non-compliance|non compliance|exceed|exceedance|breach|limit", regex=True)
combined["incident_flag"] = combined["text_blob"].str.contains("incident|shutdown|bypass|abnormal|failure|malfunction", regex=True)

summary = combined.groupby(["permit_id","report_year"], as_index=False).agg(
    cems_hit_count=("permit_id","size"),
    noncompliance_hits=("noncompliance_flag","sum"),
    incident_hits=("incident_flag","sum"),
)

if not registry.empty:
    summary = summary.merge(
        registry[["permit_id","facility_name","site_id","region_folder"]].drop_duplicates(),
        on="permit_id", how="left"
    )

summary.to_csv(out / "documentary_noncompliance_warehouse.csv", index=False)
write_json(out / "documentary_noncompliance_summary.json", {
    "rows": int(len(summary)),
    "permits": int(summary["permit_id"].nunique()),
})

manifest = manifest_base(step, [CONFIGS / "permit_registry.yml"])
manifest["inputs"] = {"cems_vocab": meta_vocab, "cems_blocks": meta_blocks}
add_artifact(manifest, out / "documentary_noncompliance_warehouse.csv")
add_artifact(manifest, out / "documentary_noncompliance_summary.json")
write_json(out / "manifest.json", manifest)
display(summary.head(20))